# PART 8 practice — one clip with Wan2.1 (Colab, free T4)

**What this does:** generates ONE short clip (about 2 seconds, 480p) from a text prompt with Wan2.1-T2V-1.3B.

**Why:** to see with your own eyes that a video model gives you a few seconds per run, and that two runs do not agree with each other. That is the gap the workflow in PART 8 has to fill.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. The first run downloads about 6 GB of weights (2–4 min) and then generates for 5–10 min. Start it, then go back to your storyboard while it runs.

If you are opening this in VS Code with the Google Colab extension, pick the Colab kernel (top right) first. Everything else is the same.

In [ ]:
!nvidia-smi -L
!pip -q install -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece

## 1. The prompt — this is the only line you decide

Write what the camera sees, in one sentence. Keep to the rules of the course: no real people's faces, no logos, no text in the image.

In [ ]:
PROMPT = "View from the window of a Mumbai suburban train slowing down as it approaches a station platform at dawn, warm light, cinematic"
NEGATIVE = "text, watermark, logo, blurry, distorted faces"
SEED = 1
NUM_FRAMES = 33      # 33 frames = about 2 seconds at 16 fps. More frames = much longer.
WIDTH, HEIGHT = 832, 480

## 2. Encode the prompt (text encoder), then free it

The text encoder of Wan2.1 (UMT5-XXL) is by far the largest part. On a T4 it does not fit next to the video model, so it runs first, its output is kept, and it is deleted before the video model is loaded.

In [ ]:
import gc, torch
from transformers import UMT5EncoderModel, AutoTokenizer

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
tok = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
enc = UMT5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=torch.float16).to("cuda")

def encode(text):
    ids = tok(text, max_length=512, padding="max_length", truncation=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        return enc(**ids).last_hidden_state

prompt_embeds = encode(PROMPT)
negative_embeds = encode(NEGATIVE)

del enc; gc.collect(); torch.cuda.empty_cache()
print("prompt encoded; text encoder freed")

## 3. Generate

This is the slow cell. Watch the progress bar; each step is one denoising pass over all frames.

In [ ]:
from diffusers import WanPipeline, AutoencoderKLWan

vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()

generator = torch.Generator("cuda").manual_seed(SEED)
frames = pipe(
    prompt_embeds=prompt_embeds,
    negative_prompt_embeds=negative_embeds,
    height=HEIGHT, width=WIDTH,
    num_frames=NUM_FRAMES,
    num_inference_steps=25,
    guidance_scale=5.0,
    generator=generator,
).frames[0]
print(len(frames), "frames")

## 4. Save and look

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video
export_to_video(frames, "practice_cut.mp4", fps=16)
Video("practice_cut.mp4", embed=True, width=640)

## 5. What to notice

1. **Length.** One run gives you 2 seconds. A 60-second video is 20–30 runs, or a different plan.
2. **Consistency.** Change `SEED` to 2 and run cells 3–4 again. The train, the light and the platform will not be the same. Nothing in the model remembers the previous clip.
3. **Control.** You decided one sentence. Everything else — camera, timing, what appears — the model decided.

The workflow in PART 8 exists because of these three things: the plan decides the cuts, a human approves the prompts before anything is generated, and the check measures the result.

Download `practice_cut.mp4` (Files panel on the left → right-click → Download). You can use it as one of the cuts in your final video; log it in `SOURCES.txt` as `Colab Wan2.1 (course notebook)`.